# Imports Libraries

In [2]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# LOAD DATA

In [3]:
df = pd.read_csv(
    "D:\ClimateGuardAI\datasets\engineered\heatwave_risk_engineered.csv"
)

print(df.head())

<>:2: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
<>:2: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
C:\Users\User\AppData\Local\Temp\ipykernel_22216\3382900828.py:2: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
  "D:\ClimateGuardAI\datasets\engineered\heatwave_risk_engineered.csv"


   latitude  longitude  temperature_celsius  wind_kph  wind_degree  \
0     24.57      77.72                 27.5      20.5          281   
1     23.33      77.80                 27.5      15.5          287   
2     22.07      78.93                 26.3      18.4          317   
3     21.86      77.93                 25.6      16.9          297   
4     22.75      77.72                 27.2      16.2          274   

   pressure_mb  precip_mm  humidity  cloud  feels_like_celsius  ...  \
0         1008        0.0        67     26                29.7  ...   
1         1008        0.0        70     19                30.0  ...   
2         1009        0.0        70     51                28.2  ...   
3         1009        0.0        76     65                27.6  ...   
4         1009        0.0        74     82                29.9  ...   

   season_Monsoon  season_Post_Monsoon  season_Summer  season_Winter  Cluster  \
0             1.0                  0.0            0.0            0.0   

# FEATURE ENGINEERING (YOUR ORIGINAL SCORING SYSTEM)

In [4]:
profile_score = {
    'Flood-Prone Climate': 35,
    'Heatwave-Prone Climate': 30,
    'Pollution-Prone Climate': 25,
    'Moderate Climate': 10
}

rainfall_score = {
    'Low': 5,
    'Medium': 20,
    'High': 35
}

heatwave_score = {
    'Safe': 5,
    'Warning': 20,
    'Critical': 35
}

anomaly_score = {
    'Normal': 5,
    'Anomalous Weather': 20
}

df['profile_score'] = df['Climate_Profile'].map(profile_score)
df['rainfall_score'] = df['rainfall_risk'].map(rainfall_score)
df['heatwave_score'] = df['heatwave_risk'].map(heatwave_score)
df['anomaly_score'] = df['Status'].map(anomaly_score)

C:\Users\User\AppData\Local\Temp\ipykernel_22216\546084933.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['profile_score'] = df['Climate_Profile'].map(profile_score)
C:\Users\User\AppData\Local\Temp\ipykernel_22216\546084933.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['rainfall_score'] = df['rainfall_risk'].map(rainfall_score)
C:\Users\User\AppData\Local\Temp\ipykernel_22216\546084933.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, 

# TARGET VARIABLE (ML LEARNING TARGET)

In [5]:
df['Climate_Risk_Score'] = (
      df['profile_score']
    + df['rainfall_score']
    + df['heatwave_score']
    + df['anomaly_score']
).clip(0, 100)

C:\Users\User\AppData\Local\Temp\ipykernel_22216\2481553334.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Climate_Risk_Score'] = (


# FEATURES AND TARGET

In [6]:
X = df[[
    'profile_score',
    'rainfall_score',
    'heatwave_score',
    'anomaly_score'
]]

y = df['Climate_Risk_Score']

# TRAIN TEST SPLIT

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# MODEL (BASE)


In [ ]:
# rf = RandomForestRegressor(random_state=42)

NameError: name 'RandomForestRegressor' is not defined

# HYPERPARAMETER TUNING (GRID SEARCH)

In [9]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Combine X_train and y_train to drop NaNs consistently
train_data = pd.concat([X_train, y_train], axis=1)
train_data.dropna(inplace=True)

X_train_cleaned = train_data.drop(columns=y_train.name)
y_train_cleaned = train_data[y_train.name]

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train_cleaned, y_train_cleaned)

NameError: name 'rf' is not defined

# BEST MODEL

In [ ]:
best_model = grid_search.best_estimator_

print("\nBest Parameters:", grid_search.best_params_)


Best Parameters: {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}


# PREDICTION + EVALUATION

In [ ]:
# Combine X_test and y_test to drop NaNs consistently
test_data = pd.concat([X_test, y_test], axis=1)
test_data.dropna(inplace=True)

X_test_cleaned = test_data.drop(columns=y_test.name)
y_test_cleaned = test_data[y_test.name]

y_pred = best_model.predict(X_test_cleaned)

print("\nMAE:", mean_absolute_error(y_test_cleaned, y_pred))
print("R2 Score:", r2_score(y_test_cleaned, y_pred))


MAE: 0.0009129801967097427
R2 Score: 0.999752498771516


# RISK LEVEL FUNCTION

In [ ]:
def risk_level(score):
    if score < 30:
        return 'Low'
    elif score < 60:
        return 'Medium'
    elif score < 80:
        return 'High'
    else:
        return 'Critical'

# NEW SAMPLE PREDICTION

In [ ]:
new_data = pd.DataFrame({
    'profile_score': [30],
    'rainfall_score': [35],
    'heatwave_score': [20],
    'anomaly_score': [20]
})

pred = best_model.predict(new_data)

print("\nPredicted Climate Risk Score:", pred[0])
print("Risk Level:", risk_level(pred[0]))


Predicted Climate Risk Score: 75.0
Risk Level: High


# SAVE MODEL

In [ ]:
save_path = "D:\ClimateGuardAI\backend\ml\artifacts"

os.makedirs(save_path, exist_ok=True)

with open(os.path.join(save_path, "climate_risk_model.pkl"), "wb") as f:
    pickle.dump(best_model, f)

print("\nModel saved successfully!")

output_dir = "D:\ClimateGuardAI\datasets\engineered"

os.makedirs(output_dir, exist_ok=True)

output_csv_path = os.path.join(output_dir, "climate_risk_scored_output.csv")

df.to_csv(output_csv_path, index=False)

print("\nOutput CSV saved successfully at:")
print(output_csv_path)


Model saved successfully!
